In [1]:
import numpy as np
import pyscf
from pyscf import gto, scf, lib, mcscf
from mrh.my_pyscf.mcscf.lasscf_o0 import LASSCF
from mrh.exploratory.unitary_cc import lasuccsd
from mrh.exploratory.unitary_cc.uccsd_sym0 import get_uccsd_op
from mrh.exploratory.citools import grad, lasci_ominus1
from lcc.lcc_solver import FCISolver_CC
from helper.util import print_list_matrix, get_sorted_excitations, cilas2f

from pathlib import Path

xyz = '''H      0.000000000000   0.000000000000   0.000000000000
H      1.000000000000   0.000000000000   0.000000000000
H      0.273746762116   2.195450598147   0.100000000000
H      1.232912762116   1.895450598147  -0.100000000000'''
# Initializing the molecule with RHF
#===================================
norb = 4
ncas_f = (2,2)
nelecas_f = (2,2)
spin_sub_f = (1,1)
frag_atom_list = ((0,1),(2,3))

mol = gto.M (atom = xyz, basis = 'sto-3g', output='h4_sto3g.log',
    verbose=0)
mf = scf.RHF (mol).run ()
print ("RHF energy = ", mf.e_tot)

# Running LASSCF
#===================================
las = LASSCF (mf, ncas_f, nelecas_f, spin_sub=spin_sub_f, verbose=3)
mo_loc = las.localize_init_guess (frag_atom_list, mf.mo_coeff)
las.kernel (mo_loc)
print ("LASSCF energy = ", las.e_tot)
las_ci0_f = cilas2f(las.ci, ncas_f, nelecas_f)  

a_idxs = np.array([[5,1]])
i_idxs = np.array([[6,0]])

#Computing energy through the LAS-UCC kernel using selected excitations
#==========================================================================================

lasci_ominus1.GLOBAL_MAX_CYCLE = 15000
fcisolver = lasuccsd.FCISolver_USCC(mol, a_idxs, i_idxs)
# mc_uscc.fcisolver.norb_f = ncas_f
# mc_uscc.kernel(ci0=las_ci0_f)
for a, i in zip (a_idxs, i_idxs):
    print ("a, i = ", a, i)
    errstr = 'a,i={},{} breaks sz symmetry'.format (a, i)
    #print ("SV sum orb = ",np.sum(a // norb))
    #print ("SV errstr = ", errstr)
    assert (np.sum (a//norb) == np.sum (i//norb)), errstr

psi = lasci_ominus1.LASUCCTrialState(fcisolver, las_ci0_f, norb = 4, norb_f = [2,2], nelec=[2,2])
dpci = psi.dp_ci(las_ci0_f)


RHF energy =  -2.1097213412443336
LASSCF energy =  -2.18247984175523
a, i =  [5 1] [6 0]


In [3]:
print("DP-CI")
# print_list_matrix(dpci)
for i in range(dpci.shape[0]):
    for j in range(dpci.shape[1]):
        if abs(dpci[i,j]) > 1e-5:
            print(f"dpci[{i},{j}] = {dpci[i,j]:.6f}")


DP-CI
dpci[5,5] = 0.967632
dpci[6,6] = -0.173224
dpci[9,9] = -0.180653
dpci[10,10] = 0.032340


In [4]:
print("frag_ci")
print_list_matrix(las_ci0_f)

frag_ci
[
  [
    [[ 0.     0.     0.     0.   ]
     [ 0.     0.984 -0.     0.   ]
     [ 0.    -0.    -0.176  0.   ]
     [ 0.     0.     0.     0.   ]]
  ]
  [
    [[ 0.     0.     0.     0.   ]
     [ 0.    -0.983 -0.     0.   ]
     [ 0.    -0.     0.184  0.   ]
     [ 0.     0.     0.     0.   ]]
  ]
]
